In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-v4-pro")

if not DEEPSEEK_API_KEY:
    raise RuntimeError("DEEPSEEK_API_KEY not found in environment or .env file")

llm = ChatDeepSeek(
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
    temperature=0,
    max_tokens=1500,
)

print(f"DeepSeek client initialized with model: {DEEPSEEK_MODEL}")

DeepSeek client initialized with model: deepseek-chat


In [2]:
response = await llm.ainvoke("Ответь одной короткой фразой: подключение к DeepSeek успешно?")
response

AIMessage(content='Да, успешно.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 22, 'total_tokens': 27, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 22}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '63f482e4-8105-4c0c-b5a3-0fe83b491e08', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ebb9d-603c-7733-8bd2-8bff30c70e0d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 5, 'total_tokens': 27, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

In [3]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

def get_deepseek_llm(
    model: str | None = None,
    temperature: float = 0.0,
    max_tokens: int = 1500,
) -> ChatDeepSeek:
    load_dotenv()

    api_key = os.getenv("DEEPSEEK_API_KEY")
    model_name = model or os.getenv("DEEPSEEK_MODEL", "deepseek-chat")

    if not api_key:
        raise RuntimeError("DEEPSEEK_API_KEY not found in environment or .env file")

    return ChatDeepSeek(
        model=model_name,
        api_key=api_key,
        temperature=temperature,
        max_tokens=max_tokens,
    )

In [4]:
llm = get_deepseek_llm()
response = await llm.ainvoke("Дай мне адрес официального сайта российской лизинговой компании Лизинг-трейд")
print(response.content)

К сожалению, я не могу предоставить адрес официального сайта компании «Лизинг-трейд», так как не имею доступа к актуальным базам данных компаний и не могу гарантировать, что информация, которой я располагаю, является верной на текущий момент.

**Рекомендую вам самостоятельно найти официальный сайт следующим образом:**

1.  **Используйте поисковую систему** (Яндекс, Google). Введите точное название: «Лизинг-трейд официальный сайт».
2.  **Обратите внимание на домен.** Официальные сайты российских компаний часто используют домены в зонах `.ru`, `.рф`, `.com` или `.su`. Будьте осторожны с сайтами, которые выглядят как копии, но имеют необычные домены (например, `.top`, `.xyz`).
3.  **Проверьте информацию на сайте.** Настоящий официальный сайт должен содержать:
    *   Юридический и фактический адрес компании.
    *   ИНН, ОГРН.
    *   Контактные телефоны и email.
    *   Лицензии (если требуются для лизинговой деятельности).
4.  **Используйте сервисы проверки контрагентов.** Вы можете вве

In [5]:
from typing import TypedDict, Any

from typing import TypedDict, Any

class SpiderState(TypedDict, total=False):
    company_name: str

    search_query: str
    search_results: list[dict]
    search_page_markdown: str

    selected_site_url: str
    selected_site_reason: str
    selected_site_confidence: float

    homepage_url: str
    homepage_title: str
    homepage_markdown: str
    homepage_internal_links: list[dict]

    adaptive_query: str
    adaptive_confidence: float
    adaptive_coverage_stats: dict[str, Any]
    adaptive_crawled_urls: list[str]
    adaptive_top_docs: list[dict]

    errors: list[str]

In [6]:
from pydantic import BaseModel, Field

class OfficialSiteDecision(BaseModel):
    official_url: str | None = Field(description="Most likely official corporate website")
    reason: str = Field(description="Why this domain looks like the official website")
    confidence: float = Field(description="Confidence from 0.0 to 1.0")

In [7]:
import json
import subprocess
import sys

def run_crawl_bridge(mode: str, url: str) -> dict:
    proc = subprocess.run(
        [sys.executable, "crawl_bridge.py", mode, url],
        capture_output=True,
        text=True,
        encoding="utf-8",
    )

    stdout = (proc.stdout or "").strip()
    stderr = (proc.stderr or "").strip()

    if not stdout:
        raise RuntimeError(
            f"crawl_bridge returned empty stdout. "
            f"returncode={proc.returncode}; stderr={stderr}"
        )

    try:
        data = json.loads(stdout)
    except json.JSONDecodeError:
        raise RuntimeError(
            "crawl_bridge returned non-JSON output.\n"
            f"RETURN CODE: {proc.returncode}\n"
            f"STDOUT:\n{stdout}\n\n"
            f"STDERR:\n{stderr}"
        )

    if not data.get("ok"):
        raise RuntimeError(data.get("error", "crawl_bridge returned error"))

    return data

In [8]:
from urllib.parse import quote

BLACKLIST_SEARCH_DOMAINS = {
    "yandex.ru", "ya.ru", "yabs.yandex.ru"
}


async def search_company_node(state: dict) -> dict:
    company_name = state["company_name"]
    search_query = f"{company_name} официальный сайт"
    search_url = f"https://yandex.ru/search/?text={quote(search_query)}"

    try:
        result = run_crawl_bridge("search", search_url)
    except Exception as e:
        return {
            "errors": state.get("errors", []) + [f"search_company_node failed: {e}"]
        }

    return {
        "search_query": search_query,
        "search_url": search_url,
        "search_results": result.get("search_results", []),
        "search_page_markdown": result.get("markdown", ""),
    }

In [9]:
from pydantic import BaseModel, Field

class OfficialSiteDecision(BaseModel):
    official_url: str | None = Field(description="Most likely official corporate website")
    reason: str = Field(description="Why this site looks like the official website")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")


async def select_official_site_node(state: dict) -> dict:
    company_name = state["company_name"]
    search_results = state.get("search_results", [])

    if not search_results:
        return {
            "errors": state.get("errors", []) + ["No search results to choose from"]
        }

    llm = get_deepseek_llm()
    structured_llm = llm.with_structured_output(OfficialSiteDecision)

    prompt = f"""
Ты выбираешь официальный сайт компании по результатам поиска.

Компания:
{company_name}

Кандидаты:
{search_results}

Правила:
- Выбери официальный корпоративный сайт эмитента.
- Не выбирай yandex, moex, smart-lab, finam, investing, wikipedia, rbc, interfax и другие агрегаторы.
- Предпочитай основной домен компании.
- Если уверенности нет, верни official_url = null.
"""

    decision = await structured_llm.ainvoke(prompt)

    updates = {
        "selected_site_reason": decision.reason,
        "selected_site_confidence": decision.confidence,
    }

    if decision.official_url:
        updates["selected_site_url"] = decision.official_url
        updates["homepage_url"] = decision.official_url
    else:
        updates["errors"] = state.get("errors", []) + ["Official site not identified"]

    return updates

In [10]:
async def open_start_page_node(state: dict) -> dict:
    homepage_url = state.get("homepage_url")

    if not homepage_url:
        return {
            "errors": state.get("errors", []) + ["homepage_url is empty"]
        }

    try:
        result = run_crawl_bridge("page", homepage_url)
    except Exception as e:
        return {
            "errors": state.get("errors", []) + [f"open_start_page_node failed: {e}"]
        }

    return {
        "homepage_url": homepage_url,
        "homepage_title": result.get("title", ""),
        "homepage_markdown": result.get("markdown", ""),
        "homepage_internal_links": result.get("internal_links", []),
    }

In [11]:
async def read_homepage_node(state: SpiderState) -> dict:
    markdown = state.get("homepage_markdown", "")
    internal_links = state.get("homepage_internal_links", [])

    short_links = []
    for link in internal_links[:20]:
        short_links.append({
            "text": link.get("text", ""),
            "href": link.get("href", ""),
        })

    return {
        "homepage_internal_links": short_links,
    }

In [12]:
from crawl4ai import AsyncWebCrawler, AdaptiveCrawler, AdaptiveConfig

async def adaptive_site_search(start_url: str, query: str):
    config = AdaptiveConfig(
        confidence_threshold=0.72,
        max_pages=20,
        top_k_links=3,
        min_gain_threshold=0.08,
    )

    async with AsyncWebCrawler() as crawler:
        adaptive = AdaptiveCrawler(crawler, config=config)
        state = await adaptive.digest(
            start_url=start_url,
            query=query,
        )

        top_docs = adaptive.get_relevant_content(top_k=10)

    return {
        "confidence": adaptive.confidence,
        "coverage_stats": adaptive.coverage_stats,
        "crawled_urls": list(state.crawled_urls),
        "top_docs": top_docs,
    }

In [13]:
async def adaptive_crawl_node(state: SpiderState) -> dict:
    homepage_url = state.get("homepage_url")

    if not homepage_url:
        return {
            "errors": state.get("errors", []) + ["homepage_url is empty for adaptive crawl"]
        }

    try:
        result = run_crawl_bridge("adaptive", homepage_url)
    except Exception as e:
        return {
            "errors": state.get("errors", []) + [f"adaptive_crawl_node failed: {type(e).__name__}: {e}"]
        }

    return {
        "adaptive_query": result.get("adaptive_query", ""),
        "adaptive_confidence": result.get("adaptive_confidence"),
        "adaptive_coverage_stats": result.get("adaptive_coverage_stats", {}),
        "adaptive_crawled_urls": result.get("adaptive_crawled_urls", []),
        "adaptive_top_docs": result.get("adaptive_top_docs", []),
    }

In [14]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(SpiderState)

graph.add_node("search_company", search_company_node)
graph.add_node("select_official_site", select_official_site_node)
graph.add_node("open_start_page", open_start_page_node)
graph.add_node("adaptive_crawl", adaptive_crawl_node)

graph.add_edge(START, "search_company")
graph.add_edge("search_company", "select_official_site")
graph.add_edge("select_official_site", "open_start_page")
graph.add_edge("open_start_page", "adaptive_crawl")
graph.add_edge("adaptive_crawl", END)

spider = graph.compile()

c:\Users\GaV\AppData\Local\Programs\Python\Python312\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [27]:
state = await spider.ainvoke({
    "company_name": "Роделен",
    "errors": [],
})

state

{'company_name': 'Роделен',
 'search_query': 'Роделен официальный сайт',
 'search_results': [{'title': 'Лизинговая компания «Роделен» – услуги лизинга для...',
   'url': 'https://www.rodelen.ru/'},
  {'title': 'Контакты', 'url': 'https://www.rodelen.ru/contacts/'},
  {'title': 'Инвесторам', 'url': 'https://www.rodelen.ru/investors/'},
  {'title': 'Агентам', 'url': 'https://www.rodelen.ru/agentam/'},
  {'title': 'Услуги лизинга', 'url': 'https://www.rodelen.ru/leasing/'},
  {'title': 'Лизинг авто',
   'url': 'https://www.rodelen.ru/leasing/avtomobilej/'},
  {'title': 'Лизинговая компания "Роделен" — ВКонтакте',
   'url': 'https://vk.com/rodelen'},
  {'title': 'Лизинговая компания Роделен – Telegram',
   'url': 'https://t.me/s/Rodelen_lc'},
  {'title': 'Лизинговая компания «Роделен» — Википедия',
   'url': 'https://ru.wikipedia.org/wiki/%D0%9B%D0%B8%D0%B7%D0%B8%D0%BD%D0%B3%D0%BE%D0%B2%D0%B0%D1%8F_%D0%BA%D0%BE%D0%BC%D0%BF%D0%B0%D0%BD%D0%B8%D1%8F_%C2%AB%D0%A0%D0%BE%D0%B4%D0%B5%D0%BB%D0%B5%

In [24]:
print("=== SEARCH PAGE MARKDOWN (Yandex) ===")
md = state.get("search_page_markdown", "")
print(md[:20000])  # первые 4000 символов, чтобы не утонуть
print("\n--- length:", len(md))

=== SEARCH PAGE MARKDOWN (Yandex) ===
[](https://www.ya.ru)
# Подтвердите, что запросы отправляли вы, а не робот
Нам очень жаль, но запросы с вашего устройства похожи на автоматические. [Почему это могло произойти?](https://yandex.ru/support/smart-captcha/problems.html?form-unique_key=8858183806207563995&form-fb-hint=1.1)
Я не робот Нажмите, чтобы продолжить
[Yandex SmartCaptcha](https://yandex.cloud/ru/services/smartcaptcha?utm_source=captcha&utm_medium=chbx&utm_campaign=security)
Если у вас возникли проблемы, пожалуйста, воспользуйтесь [формой обратной связи](https://yandex.ru/support/smart-captcha/problems.html?form-unique_key=8858183806207563995&form-fb-hint=1.1)
Если вам нужно автоматически задавать запросы к Поиску, используйте [Yandex Search API v2](https://yandex.cloud/ru/docs/search-api/quickstart/v2)
8858183806207563995:1781264666


--- length: 815


In [25]:
print("\n=== HOMEPAGE MARKDOWN (lime-zaim.ru) ===")
hp_md = state.get("homepage_markdown", "")
print(hp_md[:4000])
print("\n--- length:", len(hp_md))

print("\n=== HOMEPAGE INTERNAL LINKS ===")
for link in state.get("homepage_internal_links", [])[:20]:
    print(link)


=== HOMEPAGE MARKDOWN (lime-zaim.ru) ===


--- length: 0

=== HOMEPAGE INTERNAL LINKS ===


In [26]:
print("adaptive_query =", state.get("adaptive_query"))
print("adaptive_confidence =", state.get("adaptive_confidence"))
print("adaptive_coverage_stats =", state.get("adaptive_coverage_stats"))
print("adaptive_crawled_urls =", len(state.get("adaptive_crawled_urls", [])))
print()

for i, doc in enumerate(state.get("adaptive_top_docs", []), 1):
    print("=" * 80)
    print("RANK:", i)
    print("URL:", doc.get("url"))
    print("TITLE:", doc.get("title"))
    print("SCORE:", doc.get("score"))
    print("CONTENT PREVIEW:", (doc.get("content") or "")[:500])
    print()

adaptive_query = инвесторам дивиденды акции облигации ценные бумаги купон купонная выплата дефолт неисполнение обязательств дополнительная эмиссия дополнительный выпуск раскрытие информации финансовая отчетность показателикорпоративные события инвестиции
adaptive_confidence = 0.0
adaptive_coverage_stats = {'pages_crawled': 0, 'total_content_length': 0, 'unique_terms': 0, 'total_terms': 0, 'pending_links': 0, 'confidence': 0.0, 'coverage': 0.0, 'consistency': 0.0, 'saturation': 0.0}
adaptive_crawled_urls = 0



In [ ]:
## Нужно добавить: умение находить менюшки раскрываюющиеся и открывать их
## Находить текст новости, читать ее и ассоциировать с датой
## Находить раздел для инвесторов и открывать сообщения, если ссылка ведет на файл - то открывать файл и читать его (в т.ч. через OCR)
## Нужно эту логику отсроить в виде графа: переходы между разделами, поиск новостей/файлов.
## Критери останова - не ходить кругами, но если видим какие-то новые пути, то можно и походить какое-то время.
## Интересуют разделы: раскрытие информации, пресс-центр, новости, инвесторам, инвестиции
## Запоминаем пути страниц, где уже были, смотрим на даты.

In [1]:
import requests

url = "https://www.datalab.to/api/v1/health"

response = requests.get(url)

print(response.text)

{"status":"ok"}


In [4]:
import requests

url = "https://www.datalab.to/api/v1/user_health"

headers = {"X-API-Key": "VxXnjNJ-607pAN21dzWYb6MqaUy3Vg-ES9Q4tXfEBS0"}

response = requests.get(url, headers=headers)

print(response.text)

{"status":"ok"}


In [9]:
from datalab_sdk import DatalabClient, ConvertOptions
import os
from dotenv import load_dotenv
load_dotenv()

DATALAB_API_KEY = os.getenv("DATALAB_API_KEY")

client = DatalabClient(DATALAB_API_KEY)

# With options
options = ConvertOptions(
    output_format="html",
    mode="balanced",
    paginate=True,
    max_pages = 20,
    disable_image_extraction = True
)
result = client.convert("input.pdf", options=options)
print(f"Quality score: {result.parse_quality_score}")
print(result.html)

RuntimeError: asyncio.run() cannot be called from a running event loop